# Veri Seti Keşfi ve Analizi

Bu notebook'ta doktorsitesi veri setini indirip analiz edeceğiz.

## Veri Seti Hakkında
- **Kaynak**: Huggingface - alibayram/doktorsitesi
- **İçerik**: Türkçe tıbbi soru-cevap çiftleri
- **Boyut**: 167,732 kayıt
- **Format**: Parquet
- **Lisans**: CC BY-NC 4.0 (ticari olmayan kullanım)

In [ ]:
# Gerekli kütüphaneleri yükle
import pandas as pd
from datasets import load_dataset
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Veri Setini İndirme

Huggingface datasets kütüphanesi ile veri setini yüklüyoruz.

In [ ]:
# Veri setini yükle
dataset = load_dataset("alibayram/doktorsitesi")

# Train split'ini DataFrame'e çevir
df = dataset['train'].to_pandas()

print(f"Toplam kayıt sayısı: {len(df)}")
print(f"\nSütunlar: {df.columns.tolist()}")

In [ ]:
# İlk birkaç kaydı incele
df.head()

## Veri Analizi

Veri setinin yapısını ve içeriğini daha detaylı inceleyelim.

In [ ]:
# Temel istatistikler
print("Veri Seti Bilgileri:")
print(f"Toplam soru-cevap çifti: {len(df)}")
print(f"Eksik değerler:\n{df.isnull().sum()}")
print(f"\nBenzersiz doktor sayısı: {df['doctor_title'].nunique()}")
print(f"Benzersiz uzmanlık alanı sayısı: {df['doctor_speciality'].nunique()}")

In [ ]:
# En çok soru cevaplanan uzmanlık alanları
top_specialities = df['doctor_speciality'].value_counts().head(15)

plt.figure(figsize=(12, 6))
top_specialities.plot(kind='barh')
plt.title('En Çok Soru Cevaplanan 15 Uzmanlık Alanı')
plt.xlabel('Soru Sayısı')
plt.ylabel('Uzmanlık Alanı')
plt.tight_layout()
plt.show()

print("\nEn popüler 10 uzmanlık alanı:")
print(top_specialities.head(10))

In [ ]:
# Soru ve cevap uzunluklarını analiz et
df['question_length'] = df['question_content'].str.len()
df['answer_length'] = df['question_answer'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['question_length'].dropna(), bins=50, edgecolor='black')
axes[0].set_title('Soru Uzunlukları Dağılımı')
axes[0].set_xlabel('Karakter Sayısı')
axes[0].set_ylabel('Frekans')

axes[1].hist(df['answer_length'].dropna(), bins=50, edgecolor='black', color='orange')
axes[1].set_title('Cevap Uzunlukları Dağılımı')
axes[1].set_xlabel('Karakter Sayısı')
axes[1].set_ylabel('Frekans')

plt.tight_layout()
plt.show()

print(f"Ortalama soru uzunluğu: {df['question_length'].mean():.0f} karakter")
print(f"Ortalama cevap uzunluğu: {df['answer_length'].mean():.0f} karakter")

## Örnek Soru-Cevap İncelemeleri

Veri setinden rastgele örnekler görelim.

In [ ]:
# Rastgele 3 örnek göster
samples = df.sample(3)

for idx, row in samples.iterrows():
    print("="*80)
    print(f"Doktor: {row['doctor_title']}")
    print(f"Uzmanlık: {row['doctor_speciality']}")
    print(f"\nSoru: {row['question_content'][:300]}..." if len(row['question_content']) > 300 else f"\nSoru: {row['question_content']}")
    print(f"\nCevap: {row['question_answer'][:300]}..." if len(row['question_answer']) > 300 else f"\nCevap: {row['question_answer']}")
    print()

## Veri Ön İşleme

RAG sistemi için veri setini hazırlıyoruz.

In [ ]:
# Eksik değerleri temizle
df_clean = df.dropna(subset=['question_content', 'question_answer'])

# Çok kısa soru/cevapları filtrele (en az 20 karakter)
df_clean = df_clean[
    (df_clean['question_length'] >= 20) & 
    (df_clean['answer_length'] >= 20)
]

print(f"Temizleme sonrası kayıt sayısı: {len(df_clean)}")
print(f"Kaldırılan kayıt sayısı: {len(df) - len(df_clean)}")

In [ ]:
# RAG için doküman formatı oluştur
# Her soru-cevap çiftini birleştirip metadata ekle

df_clean['document'] = (
    "Soru: " + df_clean['question_content'] + 
    "\n\nCevap: " + df_clean['question_answer']
)

# Metadata dict oluştur
df_clean['metadata'] = df_clean.apply(
    lambda row: {
        'doctor_title': row['doctor_title'],
        'speciality': row['doctor_speciality'],
        'question': row['question_content'][:100]  # İlk 100 karakter
    }, axis=1
)

print("Doküman formatı hazır!")
print(f"\nÖrnek doküman:\n{df_clean['document'].iloc[0][:400]}...")

In [ ]:
# Veri setini kaydet
df_clean.to_parquet('../data/processed_medical_qa.parquet', index=False)
print("Veri seti kaydedildi: data/processed_medical_qa.parquet")

## Özet

Bu notebook'ta:
1. Doktorsitesi veri setini Huggingface'den indirdik
2. Veri setinin yapısını ve içeriğini analiz ettik
3. Uzmanlık alanlarını ve soru-cevap uzunluklarını görselleştirdik
4. Veri temizleme ve ön işleme yaptık
5. RAG sistemi için doküman formatı oluşturduk

Sonraki adımda, bu veriyi kullanarak embedding oluşturup vector database'e yükleyeceğiz.